# Room Redesign — SDXL + ControlNet

**Input:** 1 ảnh phòng + 3 lựa chọn: **loại phòng**, **phong cách**, **giữ layout hay không**

**Output:** ảnh phòng render lại với nội thất đúng loại phòng bạn chọn, theo đúng phong cách

**Nguyên tắc:** *loại phòng luôn thắng*. Đồ đạc trong ảnh gốc bị xoá hết, thay bằng đồ đúng loại
phòng bạn chọn. Ảnh gốc chỉ còn dùng để giữ kiến trúc — tường, cửa sổ, tỉ lệ phòng.

| `keep_layout` | Giữ gì |
|---|---|
| **True** | Cửa sổ, đường chân tường, góc tường, chiều cao trần đứng đúng chỗ. Dùng khi ảnh gốc là phòng thật của khách. |
| **False** | Chỉ giữ đại khái hình dạng phòng, model được đổi cả vị trí cửa sổ / mảng tường. Ảnh thường đẹp hơn nhưng không còn là phòng đó nữa. |

**Kỹ thuật:** SDXL + checkpoint nội thất (RealVisXL / Juggernaut XL) + ControlNet-MLSD
(chỉ trích đường thẳng kiến trúc, nên đồ đạc cũ không dính vào control map).

> **Bắt buộc:** Runtime > Change runtime type > **GPU** (T4 là đủ).
> Thời gian: ~60-120 giây/ảnh trên T4 free.


## 1. Cài đặt thư viện

In [ ]:
!pip install -q --upgrade diffusers transformers accelerate safetensors peft
!pip install -q controlnet_aux opencv-python-headless


## 2. Import & load model

Lần đầu chạy sẽ tải ~8-10 GB (SDXL + 2 ControlNet), mất 3-6 phút.

- `BASE_MODEL`: checkpoint SDXL. RealVisXL V5 và Juggernaut XL v9 đều mạnh về nội thất/kiến trúc; `stabilityai/stable-diffusion-xl-base-1.0` là bản gốc (an toàn nhất nhưng ảnh "AI" hơn).
- `use_small_controlnet`: bản ControlNet rút gọn của diffusers, nhẹ VRAM hơn hẳn — nên bật trên T4 free. Tắt nếu chạy trên A100/L4 để có độ bám sát cao hơn.


In [ ]:
#@title Load SDXL + ControlNet { display-mode: "form" }
BASE_MODEL = "SG161222/RealVisXL_V5.0"  #@param ["SG161222/RealVisXL_V5.0", "RunDiffusion/Juggernaut-XL-v9", "stabilityai/stable-diffusion-xl-base-1.0"]
use_small_controlnet = True  #@param {type:"boolean"}
#@markdown `load_depth_controlnet`: ControlNet-Depth giữ khối 3D của đồ đạc cũ. Cách làm hiện tại luôn
#@markdown xoá đồ cũ nên **không cần depth** -> để tắt cho nhẹ VRAM. Chỉ bật nếu bạn tự sửa code để giữ đồ.
load_depth_controlnet = False  #@param {type:"boolean"}

import gc
import numpy as np
import torch
from PIL import Image
from diffusers import (
    StableDiffusionXLControlNetPipeline,
    ControlNetModel,
    AutoencoderKL,
    UniPCMultistepScheduler,
)

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32
if device == "cpu":
    print("CẢNH BÁO: không thấy GPU. Runtime > Change runtime type > GPU, rồi chạy lại.")

if use_small_controlnet:
    CANNY_ID = "diffusers/controlnet-canny-sdxl-1.0-small"
    DEPTH_ID = "diffusers/controlnet-depth-sdxl-1.0-small"
else:
    CANNY_ID = "diffusers/controlnet-canny-sdxl-1.0"
    DEPTH_ID = "diffusers/controlnet-depth-sdxl-1.0"

# ---- depth estimator (chỉ cần khi xử lý phòng đã có đồ) ----
depth_estimator = None
if load_depth_controlnet:
    from controlnet_aux import MidasDetector
    print("Đang tải depth estimator (MiDaS)...")
    depth_estimator = MidasDetector.from_pretrained("lllyasviel/Annotators")

# ---- MLSD line detector: chi bat duong THANG DAI (tuong, cua so, tran)
#      -> dung cho che do doi cong nang phong, vi do dac cu bi loai khoi control map
from controlnet_aux import MLSDdetector
print("Đang tải MLSD line detector...")
mlsd_detector = MLSDdetector.from_pretrained("lllyasviel/Annotators")

# ---- ControlNet ----
print(f"Đang tải ControlNet-Canny: {CANNY_ID}")
controlnet_canny = ControlNetModel.from_pretrained(CANNY_ID, torch_dtype=dtype, variant=None)

controlnet_depth = None
if load_depth_controlnet:
    print(f"Đang tải ControlNet-Depth: {DEPTH_ID}")
    controlnet_depth = ControlNetModel.from_pretrained(DEPTH_ID, torch_dtype=dtype, variant=None)

# ---- VAE: bản fp16-fix, tránh ảnh ra đen/NaN khi chạy fp16 trên T4 ----
print("Đang tải VAE fp16-fix...")
vae = AutoencoderKL.from_pretrained("madebyollin/sdxl-vae-fp16-fix", torch_dtype=dtype)

# ---- Pipeline: nạp danh sách 2 controlnet, lúc gọi sẽ chọn dùng 1 hay 2 ----
controlnets = [controlnet_canny] + ([controlnet_depth] if controlnet_depth is not None else [])

print(f"Đang tải SDXL: {BASE_MODEL}")
def _load_pipe(variant):
    return StableDiffusionXLControlNetPipeline.from_pretrained(
        BASE_MODEL,
        controlnet=controlnets,
        vae=vae,
        torch_dtype=dtype,
        use_safetensors=True,
        variant=variant,
    )

# Nhieu checkpoint community khong up ban fp16 rieng -> thu fp16 truoc, khong co thi lay ban full
try:
    pipe = _load_pipe("fp16" if dtype == torch.float16 else None)
except Exception as e:
    print(f"  Khong co variant fp16 ({type(e).__name__}), tai ban day du...")
    pipe = _load_pipe(None)
pipe.scheduler = UniPCMultistepScheduler.from_config(pipe.scheduler.config)

# Tiết kiệm VRAM trên T4 16GB: offload từng module sang CPU khi không dùng
pipe.enable_model_cpu_offload()

# VAE slicing/tiling: diffusers moi bo 2 method nay o cap pipeline, chuyen xuong pipe.vae
for _name in ("enable_slicing", "enable_tiling"):
    if hasattr(pipe.vae, _name):
        getattr(pipe.vae, _name)()
    elif hasattr(pipe, f"enable_vae_{_name.split('_')[1]}"):
        getattr(pipe, f"enable_vae_{_name.split('_')[1]}")()

gc.collect()
torch.cuda.empty_cache() if device == "cuda" else None

CONTROL_ORDER = ["canny"] + (["depth"] if controlnet_depth is not None else [])
print(f"\nModel sẵn sàng. ControlNet đang nạp: {CONTROL_ORDER}")


## 3. Room type & Style
- `ROOM_TYPES`: loại phòng -> mô tả tiếng Anh + nội thất đặc trưng (giúp model đặt đúng đồ vào đúng phòng).
- `STYLE_PROMPTS`: phong cách -> mô tả vật liệu / màu / ánh sáng.
- Prompt cuối = style + room + nội thất + quality tag. Thêm loại phòng / style mới thì thêm 1 dòng vào dict, và thêm vào danh sách `#@param` ở bước 5.


In [ ]:
import random
import re

# ---------- 3.1 Nhóm không gian ----------
# 3 nhóm bao trùm. Mỗi nhóm có danh sách loại / style / bảng màu RIÊNG, lấy đúng theo
# 6 file JSON được cung cấp. Garden không có "loại phòng" - chỉ chọn style và màu.
SPACE_INTERIOR = "Interior"
SPACE_EXTERIOR = "Exterior"
SPACE_GARDEN = "Garden"

SPACES = {
    SPACE_INTERIOR: {
        "Phòng khách"             : ("living room"                   , "large sofa, coffee table, armchair, area rug, floor lamp, framed wall art, curtains"),
        "Phòng ngủ"               : ("bedroom"                       , "double bed with headboard, bedding, nightstands, bedside lamps, wardrobe, area rug, curtains"),
        "Phòng ăn"                : ("dining room"                   , "dining table, dining chairs, pendant light above table, sideboard, centerpiece"),
        "Phòng tắm"               : ("bathroom"                      , "bathtub, toilet, vanity with mirror, walk-in shower, wall tiles, towels"),
        "Bếp"                     : ("kitchen"                       , "kitchen cabinets, stone countertop, kitchen island, bar stools, tile backsplash, range hood"),
        "Phòng chơi game"         : ("gaming room"                   , "gaming desk against the wall, desktop computer with two monitors on the desk, keyboard and mouse, racing gaming chair"),
        "Nhà hàng"                : ("restaurant interior"           , "multiple dining tables and chairs, bar counter, decorative pendant lights, banquette seating"),
        "Văn phòng tại gia"       : ("home office"                   , "desk with computer, ergonomic chair, bookshelf, task lamp, potted plants"),
        "Quán cà phê"             : ("coffee shop interior"          , "cafe counter, espresso machine, small round tables, bentwood chairs, menu board, pendant lights"),
        "Văn phòng"               : ("corporate office"              , "workstations, office desks, task chairs, meeting table, acoustic panels, carpet tiles"),
    },
    SPACE_EXTERIOR: {
        "Nhà"                     : ("house exterior"                , "front door, windows, pitched roof, driveway, front lawn"),
        "Hồ bơi"                  : ("swimming pool area"            , "swimming pool, sun loungers, parasol, pool deck, surrounding planting"),
        "Nhà siêu nhỏ"            : ("tiny house exterior"           , "compact timber cabin, large window, small deck, pitched roof, entry steps"),
        "Sân trong"               : ("courtyard exterior"            , "enclosed courtyard walls, stone paving, planting beds, feature tree, outdoor seating"),
        "Khu nghỉ dưỡng"          : ("resort exterior"               , "resort buildings, pool, sun loungers, palm planting, walkways"),
        "Gian nhà vòm"            : ("dome house exterior"           , "geodesic dome structure, curved glazing, entrance door, surrounding planting"),
        "Quán cà phê (ngoài)"     : ("coffee shop exterior"          , "shopfront glazing, cafe signage, outdoor tables and chairs, awning, planters"),
        "Nông trại"               : ("farm exterior"                 , "farm house, barn, fenced field, gravel yard, mature trees"),
        "Biệt thự"                : ("villa exterior"                , "pitched roof, tall windows, entrance porch, driveway, front lawn"),
        "Căn hộ"                  : ("apartment building exterior"   , "balconies, repeated window bays, entrance canopy, facade cladding"),
        "Nhà phố"                 : ("townhouse facade"              , "front door, windows, balcony railing, facade cladding, street level entrance"),
        "Văn phòng (ngoài)"       : ("office building facade"        , "glass curtain wall, entrance lobby, signage band, paved forecourt"),
        "Cửa hàng"                : ("retail shopfront"              , "display windows, shop signage, entrance door, awning, pavement"),
        "Cabin"                   : ("log cabin exterior"            , "timber log walls, pitched roof, porch, chimney, forest surroundings"),
        "Khách sạn"               : ("hotel exterior"                , "hotel entrance canopy, repeated window bays, signage, landscaped forecourt"),
        "Gian hàng"               : ("market stall exterior"         , "open front stall, canopy roof, counter display, signage, string lights"),
        "Nhà tranh"               : ("thatched cottage exterior"     , "thatched roof, timber windows, garden path, climbing plants, short fence"),
        "Tháp"                    : ("tower building exterior"       , "tall slender tower, vertical fins, glazed shaft, base entrance"),
    },
    # Garden không có loại cụ thể -> dùng default_subject của nhóm.
    SPACE_GARDEN: {},
}

SPACE_OPTIONS = list(SPACES)
# Gộp phẳng để phần code còn lại vẫn dùng ROOM_TYPES[loại] = (tên tiếng Anh, hạng mục)
ROOM_TYPES = {name: value for group in SPACES.values() for name, value in group.items()}
SPACE_OF = {name: space for space, group in SPACES.items() for name in group}
assert len(ROOM_TYPES) == sum(len(g) for g in SPACES.values()), "Tên loại bị trùng giữa các nhóm"
# Dropdown của Colab là danh sách phẳng nên tên phải duy nhất: "Quán cà phê" và
# "Văn phòng" có ở cả Interior lẫn Exterior, bản Exterior thêm hậu tố "(ngoài)".

NO_ROOM_TYPE = "— không áp dụng (Garden) —"


def rooms_of(space: str) -> list:
    """Danh sách loại hợp lệ của 1 nhóm. Garden trả về [] (không có loại)."""
    assert space in SPACES, f"Nhóm không hợp lệ. Chọn 1 trong: {SPACE_OPTIONS}"
    return list(SPACES[space])


# Những cụm từ phụ thuộc nhóm. Đây là lý do phải tách 3 nhóm ra: "interior",
# "fully furnished", "empty room" đều vô nghĩa với mặt tiền nhà hay sân vườn.
SPACE_TEMPLATES = {
    SPACE_INTERIOR: {
        "suffix":  "interior",
        "default_subject": ("interior room", "furniture, lighting, floor finish"),
        "quality": ("photorealistic, professional interior photography, soft natural lighting, "
                    "detailed material and surface texture, sharp focus, high resolution"),
        "complete": "fully furnished, professionally staged, all essential furniture present",
        "empty_neg": "empty room, unfurnished, bare floor, no furniture, vacant",
        "other_space_neg": "outdoors, street view",
        # chỉ Interior mới cấm thêm cửa: mặt tiền và sân vườn thì cửa/cửa sổ là hạng mục cần có
        "extra_neg": "extra doors, extra windows",
        # Liệt kê tên các loại cùng nhóm vào negative để dập prior "living room" của SDXL.
        # Exterior/Garden không cần: tên các loại cùng nhóm trùng chữ nhau quá nhiều.
        "negate_siblings": True,
    },
    SPACE_EXTERIOR: {
        "suffix":  "",
        "default_subject": ("building exterior", "facade, windows, entrance, surrounding ground"),
        "quality": ("photorealistic, professional architectural exterior photography, natural daylight, "
                    "detailed material texture, sharp focus, high resolution"),
        "complete": "finished building, landscaped surroundings, professionally staged",
        "empty_neg": "construction site, scaffolding, bare plot, rubble",
        "other_space_neg": "indoors, interior room",
        "extra_neg": "",
        "negate_siblings": False,
    },
    SPACE_GARDEN: {
        "suffix":  "",
        "default_subject": ("garden", "lawn, planting beds, garden path, mature trees and shrubs, outdoor seating"),
        "quality": ("photorealistic, professional landscape garden photography, natural daylight, "
                    "detailed foliage texture, sharp focus, high resolution"),
        "complete": "fully planted, mature healthy planting, professionally landscaped",
        "empty_neg": "bare soil, dead vegetation, scorched ground, empty lot, construction site",
        "other_space_neg": "indoors, interior room, carpet",
        "extra_neg": "",
        "negate_siblings": False,
    },
}

# Chi tiết vật liệu / ánh sáng / hoàn thiện riêng của từng loại, đi vào prompt_2.
# KHÔNG ghi màu ở đây - màu là việc của COLOR_PALETTES.
ROOM_EXTRA = {
    "Phòng chơi game":    "the computer desk setup is the main subject",
    "Phòng tắm":          "large format wall and floor tiles, glass shower screen, wall mounted fittings",
    "Bếp":                "handleless cabinet fronts, integrated appliances, under cabinet lighting",
    "Văn phòng":          "suspended ceiling lighting, glass partition, cable managed desks",
    "Văn phòng (ngoài)":  "clean glazing lines, crisp shadow, wide angle street view",
    "Hồ bơi":             "clear still water, wet deck reflection",
    "Tháp":               "wide angle upward view, open sky backdrop",
    "Cửa hàng":           "lit display window, street level view",
}

# ---------- 3.2 Phong cách ----------
# value = (look, màu mặc định)
#   look = VẬT LIỆU / HÌNH KHỐI / ÁNH SÁNG / KHÔNG KHÍ. Không màu, không tên món đồ.
#   màu mặc định = dùng khi người dùng chọn Color = "Theo style".
# Tách 2 phần để option Color thay được màu mà không phá bản sắc của style.
# Danh sách tên lấy từ Interior_style_list_full.json / exterior_style_list_full.json /
# garden_style_list_full.json (các file này chỉ có tên, prompt và description đều null).
STYLE_PROMPTS = {
    "Electric": ("electric bold style, high contrast surfaces, graphic shapes, vivid accent lighting, energetic mood",
                 "vivid saturated palette with strong contrast"),
    "Modern": ("modern style, clean straight lines, flat planes, large glazing, unfussy detailing",
               "neutral palette with dark accents"),
    "Peaceful": ("peaceful serene style, matte natural finishes, woven linen and wool texture, soft diffused daylight",
                 "warm taupe and greige walls, muted olive accents, dark walnut wood"),
    "Farmhouse": ("modern farmhouse style, shiplap walls, reclaimed wood beams, vintage patina, cozy atmosphere",
                  "warm neutral tones, aged white paint, honey oak"),
    "Clean Bright": ("clean bright style, bright even daylight, fresh and airy atmosphere, crisp detailing",
                     "crisp white walls, light oak wood"),
    "Contemporary": ("contemporary style, sleek forms, mixed textures, designer lighting, polished detailing",
                     "muted palette with one bold accent color, matte black details"),
    "Fresh Airy": ("fresh airy style, abundant daylight, light sheer fabrics, indoor greenery, breezy open atmosphere",
                   "pale airy palette, soft white and light wood"),
    "Eclectic": ("eclectic style, patterned textiles, gallery wall, mix of vintage and modern, playful energetic atmosphere",
                 "bold saturated color mix"),
    "Elegant": ("elegant style, silk and velvet textures, refined symmetry, polished surfaces",
                "soft neutral palette, subtle gold details"),
    "Minimalist": ("minimalist style, clean lines, flush surfaces, restrained decor, quiet composition",
                   "neutral monochrome palette"),
    "Minimal Tranquil": ("minimal tranquil style, soft diffused light, natural linen, zen calm, restrained decor",
                         "warm off white tones, pale wood"),
    "Cartoon": ("cartoon illustration style, thick outlines, playful stylized shapes, cel shaded, 2d render",
                "flat bold colors"),
    "Scandinavian": ("scandinavian style, cozy woven textiles, hygge atmosphere, soft natural light",
                     "light wood floor, white walls, pale neutral tones"),
    "Simple Calm": ("simple calm style, slim rounded forms, gentle even lighting, quiet restrained decor",
                    "soft beige and grey palette"),
    "Bright Soothing": ("bright soothing style, soft rounded forms, warm sunlight, comfortable relaxing atmosphere",
                        "pastel palette"),
    "Cyberpunk": ("cyberpunk style, dark glossy surfaces, holographic panels, futuristic tech details, moody atmosphere",
                  "neon pink and cyan lighting"),
    "Rustic": ("rustic style, exposed wooden beams, stone wall, rough sawn timber, woven natural textiles, aged patina",
               "earthy warm tones"),
    "Compact Calm": ("compact calm style, space saving forms, neat organised layout, efficient use of space",
                     "light neutral palette"),
    "Classic Graceful": ("classic graceful style, wall moulding, restrained ornament, crystal light fixtures, high ceiling",
                         "cream and soft blue palette"),
    "Traditional": ("traditional style, patterned textiles, symmetrical layout, warm lamp light, framed wall decor",
                    "dark stained wood, deep warm tones"),
    "Industrial": ("industrial style, exposed brick wall, metal fixtures, concrete floor, aged leather texture, edison bulb lighting",
                   "raw grey and rust tones, matte black metal"),
    "Mid-Century": ("mid century modern style, teak wood, tapered legs, geometric patterns, 1960s design",
                    "mustard and olive accents, warm walnut"),
    "Japandi": ("japandi style, japanese and scandinavian fusion, natural wood, paper screen and woven straw texture, restrained decor",
                "soft neutral tones, pale wood and charcoal accents"),
    "Bohemian": ("bohemian style, layered textiles, macrame and rattan texture, trailing plants, relaxed lived in feel",
                 "warm earthy palette with jewel accents"),
    "Dry Serene": ("dry serene style, arid desert calm, matte plaster, sun bleached texture, still quiet mood",
                   "sun bleached sand and clay tones"),
    "Romantic Lush": ("romantic lush style, floral patterns, soft draping fabric, candle warm light, abundant blooms",
                      "soft rose and cream palette"),
    "Natural Cozy": ("natural cozy style, raw wood grain, chunky knit texture, warm lamp glow, inviting mood",
                     "warm wood and oat tones"),
    "Soft Aesthetic": ("soft aesthetic style, rounded silhouettes, matte finishes, diffused glow, gentle dreamy mood",
                       "muted pastel palette"),
    "Luxury": ("luxury style, marble veining, polished metal trim, deep pile texture, statement lighting, opulent mood",
               "rich dark tones with gold accents"),
    "Tropical": ("tropical style, rattan and teak, lush palm foliage, breezy open layout, resort mood",
                 "vivid foliage green with natural wood tones"),
    "Gothic": ("gothic style, pointed arches, ribbed vaulting, ornate tracery, candlelit drama, heavy stone",
               "deep dark tones with burgundy accents"),
    "Modern Dynamic": ("modern dynamic style, angular geometry, bold diagonal lines, dramatic lighting, energetic composition",
                       "high contrast neutral palette with a vivid accent"),
    "Dramatic Timeless": ("dramatic timeless style, strong contrast, sculptural forms, moody directional light, classic proportion",
                          "deep dark tones with pale contrast"),
    "Grand Traditional": ("grand traditional style, generous proportion, carved millwork, symmetrical grandeur, formal composition",
                          "rich mahogany and deep jewel tones"),
    "Charming Cozy": ("charming cozy style, small scale detailing, soft textiles, warm intimate lighting, homely mood",
                      "warm cream and soft rose tones"),
    "Raw Modern": ("raw modern style, exposed structure, raw concrete, honest materials, stark simplicity",
                   "grey concrete and black steel tones"),
    "Warm Ornate": ("warm ornate style, decorative carving, patterned tilework, layered ornament, rich detailing",
                    "warm amber and copper tones"),
    "Natural": ("natural style, untreated wood, stone and linen texture, daylight, organic forms",
                "earth toned natural palette"),
    "Calm Meditative": ("calm meditative style, quiet symmetry, soft shadow, tactile matte surfaces, contemplative stillness",
                        "muted stone and sand tones"),
    "Lush Vibrant": ("lush vibrant style, abundant greenery, layered planting, vivid energy, rich texture",
                     "vivid foliage and saturated floral tones"),
    "Warm Breezy": ("warm breezy style, light flowing fabric, open airflow, golden afternoon light, relaxed coastal mood",
                    "warm sand and soft white tones"),
    "Sleek Structured": ("sleek structured style, precise geometry, flush joints, linear lighting, disciplined composition",
                         "monochrome palette with metallic sheen"),
    "Bright Relaxing": ("bright relaxing style, generous daylight, soft textures, easy informal layout, restful mood",
                        "light airy neutral palette"),
    "Colorful": ("colorful style, playful color blocking, graphic patterns, cheerful energy",
                 "multi color saturated palette"),
    "Clean": ("clean style, crisp edges, smooth surfaces, even lighting, orderly composition",
              "bright neutral palette"),
    "Balanced": ("balanced style, symmetrical proportion, measured rhythm, even natural light, harmonious composition",
                 "balanced warm neutral palette"),
    "Cozy Homey": ("cozy homey style, soft layered textiles, warm pools of lamp light, lived in comfort",
                   "warm honey and soft cream tones"),
    "Striking": ("striking style, bold statement forms, high contrast, dramatic focal lighting",
                 "high contrast dark and light palette"),
    "Sunny": ("sunny style, strong warm sunlight, crisp shadows, cheerful open feel",
              "sunlit warm yellow and white tones"),
    "Boho Scandinavian": ("boho scandinavian style, woven texture, light wood, relaxed layering, airy hygge mood",
                          "pale neutral palette with warm sand accents"),
    "Deep Peaceful": ("deep peaceful style, enveloping quiet, matte deep surfaces, soft dim light, restful stillness",
                      "deep muted tones with soft contrast"),
    "Soft Natural Light": ("soft natural light style, diffused daylight through sheer fabric, gentle shadow gradients, airy calm",
                           "pale warm neutral palette"),
    "Bright Airy": ("bright airy style, high ceiling, open volume, abundant daylight, light breezy feel",
                    "bright white and pale wood tones"),
    "Cozy Warm": ("cozy warm style, thick tactile textures, warm glowing light, intimate enclosure",
                  "warm amber and soft brown tones"),
    "Oriental": ("oriental style, lacquered surfaces, carved screens, silk texture, refined eastern motifs",
                 "deep red and gold tones"),
    "Colorful Relaxed": ("colorful relaxed style, easy going color mix, soft informal shapes, cheerful calm",
                         "soft multi color pastel palette"),
    "Wabi-Sabi": ("wabi sabi style, imperfect handmade surfaces, aged patina, rough plaster, quiet imperfection",
                  "muted earth tones"),
    "Neo-Classic": ("neo classic style, fluted columns, restrained classical ornament, fine proportion, formal symmetry",
                    "pale stone and soft gold tones"),
    "Art Deco": ("art deco style, geometric inlay, fluted panels, glossy lacquer, 1920s glamour",
                 "deep emerald and gold tones"),
    "Rustic Luxury": ("rustic luxury style, rough hewn timber with polished finishes, refined craftsmanship, lodge grandeur",
                      "warm timber and bronze tones"),
    "Colorful Exotic": ("colorful exotic style, global patterns, carved detailing, layered textiles, vibrant faraway mood",
                        "saturated jewel tone palette"),
    "Simple Airy": ("simple airy style, light open volume, plain surfaces, gentle daylight",
                    "pale neutral palette"),
    "Rustic Lively": ("rustic lively style, rough natural materials, cheerful informal arrangement, warm busy energy",
                      "warm earthy palette with bright accents"),
    "Brutalist": ("brutalist style, massive concrete forms, board formed texture, heavy geometry, stark monumentality",
                  "raw grey concrete tones"),
    "Spanish Revival": ("spanish revival style, stucco walls, clay roof tiles, arched openings, wrought iron detailing",
                        "warm whitewash and terracotta tones"),
    "Victorian": ("victorian style, ornate trim, bay windows, decorative gables, fine spindle work",
                  "deep heritage tones with cream trim"),
    "Zen": ("zen style, raked gravel, simple timber frame, quiet asymmetry, meditative restraint",
            "muted stone and moss tones"),
    "Chinese": ("chinese traditional style, upturned eaves, timber joinery, lattice screens, courtyard symmetry",
                "deep red and dark timber tones"),
    "Cottage": ("cottage style, small scale charm, timber windows, climbing plants, homely informality",
                "soft pastel and whitewash tones"),
    "French": ("french style, mansard roof, tall shuttered windows, wrought iron balcony, refined proportion",
               "pale limestone and soft grey tones"),
    "Italianate": ("italianate style, wide eaves with brackets, tall arched windows, stucco facade, classical cornice",
                   "warm ochre and cream tones"),
    "Japanese": ("japanese style, timber post and beam, deep eaves, paper screens, engawa veranda, quiet restraint",
                 "natural timber and off white tones"),
    "Mediterranean": ("mediterranean style, whitewashed masonry, arched openings, clay tiles, sun drenched courtyard",
                      "whitewash and sea blue tones"),
    "Colonial": ("colonial style, symmetrical facade, columned porch, shuttered windows, formal proportion",
                 "soft white and deep shutter tones"),
    "Mid Century": ("mid century style, long horizontal massing, post and beam, wide overhangs, large glazing, 1960s modernism",
                    "warm wood with muted accent tones"),
    "Retro": ("retro style, nostalgic mid century shapes, rounded corners, playful graphic detailing",
              "warm retro palette of mustard and teal"),
    "Cottagecore": ("cottagecore style, romantic rural charm, climbing roses, handmade rustic detailing, soft pastoral mood",
                    "soft floral and cream tones"),
    "Desert": ("desert style, adobe plaster, sculptural cacti, sun baked surfaces, arid open mood",
               "sun bleached sand and clay tones"),
    "English Classic": ("english classic style, clipped hedges, formal symmetry, brick and stone detailing, heritage refinement",
                        "deep foliage and warm brick tones"),
    "Midcentury": ("midcentury modern style, long horizontal lines, geometric planting beds, breeze block screens, 1960s optimism",
                   "warm wood with muted accent tones"),
    "Minimal Nature": ("minimal nature style, restrained natural materials, simple planting rhythm, quiet organic calm",
                       "muted natural foliage and stone tones"),
    "Balinese": ("balinese style, thatched roof, carved timber, water features, tropical resort serenity",
                 "dark timber and lush foliage tones"),
    "Urban Courtyard": ("urban courtyard style, enclosed paved space, vertical planting, compact city oasis, soft screening",
                        "warm grey paving with foliage tones"),
    "Vertical+Green": ("vertical planting style, living plant wall, layered vertical foliage, dense leafy texture",
                       "rich foliage green tones"),
    "Coastal": ("coastal style, weathered timber, breezy open frames, sea air light, relaxed shoreline mood",
                "soft white and ocean blue tones"),
    "Forest Retreat": ("forest retreat style, tall trees, dappled shade, natural timber, secluded woodland calm",
                       "deep forest green and bark tones"),
    "Balanced Elegant": ("balanced elegant style, refined symmetry, graceful proportion, soft even lighting, polished restraint",
                         "soft neutral palette with muted accents"),
}

# Style nào dùng được cho nhóm nào - theo đúng 3 file JSON.
STYLE_BY_SPACE = {
    "Interior": ["Electric", "Modern", "Peaceful", "Farmhouse", "Clean Bright", "Contemporary", "Fresh Airy", "Eclectic", "Elegant", "Minimalist", "Minimal Tranquil", "Cartoon", "Scandinavian", "Simple Calm", "Bright Soothing", "Cyberpunk", "Rustic", "Compact Calm", "Classic Graceful", "Traditional", "Industrial", "Mid-Century", "Japandi", "Bohemian", "Dry Serene", "Romantic Lush", "Natural Cozy", "Soft Aesthetic", "Luxury", "Tropical", "Gothic", "Modern Dynamic", "Dramatic Timeless", "Grand Traditional", "Charming Cozy", "Raw Modern", "Warm Ornate", "Natural", "Calm Meditative", "Lush Vibrant", "Warm Breezy", "Sleek Structured", "Bright Relaxing", "Colorful", "Clean", "Balanced", "Cozy Homey", "Striking", "Sunny", "Boho Scandinavian", "Deep Peaceful", "Soft Natural Light", "Bright Airy", "Cozy Warm", "Oriental", "Colorful Relaxed", "Wabi-Sabi", "Neo-Classic", "Art Deco", "Rustic Luxury", "Colorful Exotic", "Simple Airy", "Rustic Lively"],
    "Exterior": ["Traditional", "Peaceful", "Clean Bright", "Fresh Airy", "Simple Calm", "Bright Soothing", "Classic Graceful", "Dry Serene", "Romantic Lush", "Minimal Tranquil", "Natural Cozy", "Modern", "Minimalist", "Soft Aesthetic", "Scandinavian", "Brutalist", "Spanish Revival", "Tropical", "Victorian", "Luxury", "Zen", "Art Deco", "Chinese", "Contemporary", "Cottage", "French", "Gothic", "Italianate", "Japanese", "Mediterranean", "Rustic", "Colonial", "Mid Century", "Charming Cozy", "Raw Modern", "Warm Ornate", "Natural", "Calm Meditative", "Lush Vibrant", "Warm Breezy", "Sleek Structured", "Colorful", "Clean", "Balanced", "Deep Peaceful", "Compact Calm", "Colorful Relaxed", "Cozy Warm", "Bright Airy", "Soft Natural Light", "Rustic Luxury", "Boho Scandinavian", "Sunny", "Striking", "Cozy Homey", "Bright Relaxing", "Colorful Exotic", "Simple Airy", "Rustic Lively", "Modern Dynamic", "Dramatic Timeless", "Grand Traditional"],
    "Garden": ["Farmhouse", "Scandinavian", "Romantic Lush", "Peaceful", "Minimal Tranquil", "Compact Calm", "Retro", "Modern", "Bohemian", "Luxury", "Contemporary", "Zen", "Cottagecore", "Desert", "English Classic", "Mediterranean", "Midcentury", "Minimal Nature", "Minimalist", "Rustic", "Tropical", "Balinese", "Urban Courtyard", "Vertical+Green", "Coastal", "Forest Retreat", "Colorful Relaxed", "Clean Bright", "Balanced Elegant", "Fresh Airy", "Bright Soothing", "Deep Peaceful", "Simple Calm", "Natural Cozy"],
}

# ---------- 3.3 Bảng màu ----------
COLOR_BY_STYLE = "Theo style"      # dùng màu mặc định của style
COLOR_RANDOM = "Surprise Me"       # random 1 bảng, khoá theo seed nên lặp lại được

COLOR_PALETTES = {
    "Muted Form": "muted greige, warm beige, taupe and charcoal palette",
    "Millennial Gray": "light grey, mid grey and charcoal palette, cool neutral tones",
    "Cozy Beige": "cozy beige, sand, tan and soft brown palette",
    "Earth Calm": "earthy sage green, warm sand and clay palette",
    "Misty Garden": "misty pale blue, sage green, soft grey with near black accents",
    "Antique Sage": "antique sage green, olive, off white and grey palette",
    "Ocean Mist": "pale ocean blue, soft sky blue with deep charcoal accents",
    "Twilight Blues": "deep navy, twilight blue and bright azure palette",
    "Velvet Dusk": "deep plum, aubergine, mauve and near black palette",
    "Sunbaked Clay": "soft peach, coral and sunbaked red clay palette",
    "Terracotta Mirage": "terracotta, warm ochre, tan and rust palette",
    "Peach Orchard": "soft peach, apricot, blush and cream palette",
    "Blush Mood": "blush pink, dusty rose and pale cream palette",
    "Pastel Breeze": "pale pastel mint, powder blue, blush and ivory palette",
    "Violet Dream": "violet, orchid purple, lilac and lavender palette",
    "Retro Pop": "retro pop palette, tomato red, teal, mustard and cream",
    "Urban Veil": "urban grey, smoke, slate and off white palette",
    "Earthy Harmony": "harmonised earth tones, warm clay, olive and oat palette",
    "Earthy Hues": "layered earth hues, ochre, umber and warm sand palette",
    "Sand Serenity": "serene sand, pale dune and warm ivory palette",
    "Terra Coast": "coastal terracotta, warm sand and soft sea blue palette",
    "Forest Hues": "deep forest green, moss and bark palette",
    "Tropic Bloom": "tropical bloom palette, vivid fuchsia, hibiscus red and lush green",
    "Emerald Gem": "rich emerald green, jade and deep teal palette",
    "Turquoise Lagoon": "turquoise lagoon palette, aqua, teal and pale sand",
    "Retro Carnival": "retro carnival palette, cherry red, sunny yellow, teal and cream",
    "Vintage Pop": "vintage pop palette, faded orange, teal, mustard and off white",
}

# Bảng màu nào dùng được cho nhóm nào - theo Interiorcolor / enterior_color /
# garden_color _list_full.json. "Surprise Me" có trong cả 3 file.
COLOR_BY_SPACE = {
    "Interior": ["Surprise Me", "Muted Form", "Millennial Gray", "Cozy Beige", "Earth Calm", "Misty Garden", "Antique Sage", "Ocean Mist", "Twilight Blues", "Velvet Dusk", "Sunbaked Clay", "Terracotta Mirage", "Peach Orchard", "Blush Mood", "Pastel Breeze", "Violet Dream", "Retro Pop"],
    "Exterior": ["Surprise Me", "Urban Veil", "Misty Garden", "Earthy Harmony", "Sunbaked Clay", "Earthy Hues", "Terracotta Mirage", "Sand Serenity", "Terra Coast", "Antique Sage", "Ocean Mist", "Twilight Blues"],
    "Garden": ["Surprise Me", "Forest Hues", "Tropic Bloom", "Emerald Gem", "Turquoise Lagoon", "Antique Sage", "Retro Carnival", "Earth Calm", "Vintage Pop", "Misty Garden", "Sand Serenity", "Terracotta Mirage", "Twilight Blues"],
}


def styles_of(space: str) -> list:
    return list(STYLE_BY_SPACE[space])


def colors_of(space: str) -> list:
    """Kèm "Theo style" ở đầu - lựa chọn này không có trong JSON, là mặc định của notebook."""
    return [COLOR_BY_STYLE] + list(COLOR_BY_SPACE[space])

# ---------- 3.4 Hạng mục / không gian "sai" ----------
# Đồ trong ảnh gốc bị xoá hết, nhưng dấu vết vẫn còn nên đẩy các món dễ bị model giữ
# lại nhất vào negative. Chỉ áp cho Interior: với Exterior/Garden thì "other_space_neg"
# ("indoors, interior room") đã chặn đồ trong nhà rồi, không cần liệt kê dài tốn token.
WRONG_FURNITURE = "sofa, couch, bed, dining table, bathtub, toilet, kitchen cabinets, office desk, bar counter"

# Bỏ các từ chung khi so trùng để "restaurant interior" không bị coi là trùng với
# "coffee shop interior" chỉ vì cùng chữ "interior".
ROOM_NAME_STOPWORDS = {"interior", "room", "exterior", "garden", "area", "building"}

# Negative riêng cho từng loại: những kiểu không gian mà model hay render nhầm sang.
# Phòng game rất dễ ra home cinema, phải cấm thẳng. Phần này đi vào negative encoder
# thứ nhất - chỗ đó còn dư token.
ROOM_NEGATIVE = {
    "Phòng chơi game":    "home theater, television, tv set, speakers, media console, soundbar",
    "Phòng tắm":          "kitchen sink, laundry",
    "Bếp":                "pub, nightclub",
    "Văn phòng tại gia":  "cubicle, call center",
    "Hồ bơi":             "theme park, public bathhouse",
    "Gian hàng":          "supermarket aisle, warehouse",
    "Nhà tranh":          "skyscraper, shopping mall",
    "Nhà siêu nhỏ":       "mansion, sprawling estate",
}

# ---------- 3.5 Quality tag & negative prompt ----------
# style vẽ minh hoạ thì không cần photorealistic -> tag riêng
NON_PHOTO_STYLES = {"Cartoon"}
NON_PHOTO_TAGS = "high quality illustration, clean vector render, detailed"

# Giữ gọn: encoder này còn phải chứa ROOM_NEGATIVE + ràng buộc nhóm. Đã bỏ các cặp
# trùng ý ("jpeg artifacts" ~ "low quality", "crooked lines" ~ "warped walls").
NEGATIVE_PROMPT = (
    "blurry, low quality, distorted, deformed furniture, watermark, text, "
    "unrealistic proportions, warped walls, people, "
    "duplicate objects, floating furniture, oversaturated"
)

# Từ đồng nghĩa: phòng khách cần "sofa" thì negative cũng phải bỏ "couch",
# nếu không prompt đòi sofa mà negative cấm couch -> model rối, sofa ra méo.
SYNONYMS = {
    "sofa": {"couch"},         "couch": {"sofa"},
    "bed": {"mattress"},       "mattress": {"bed"},
    "desk": {"workstation", "workstations"},
    "counter": {"countertop"}, "countertop": {"counter"},
}


def _words(text: str) -> set:
    words = set(re.findall(r"[a-z]+", text.lower()))
    return words | {syn for w in words for syn in SYNONYMS.get(w, ())}


# Hạng mục ĐỊNH DANH - món mà thiếu nó thì không gian thành loại khác.
# Mặc định lấy 2 phần tử đầu của danh sách hạng mục, nhưng với vài loại thì 2 món đầu
# KHÔNG phải thứ định danh:
#   - phòng game: bàn + ghế = định nghĩa của phòng làm việc. Định danh là máy tính + ghế.
#   - văn phòng tại gia / văn phòng: bàn + ghế quá chung.
ROOM_MUST_HAVE = {
    "Phòng chơi game":   "a desktop computer with two monitors and a racing gaming chair",
    "Văn phòng tại gia": "desk with a computer, bookshelf",
    "Văn phòng":         "rows of workstations, meeting table",
}


def subject_of(space: str, room_type=None) -> tuple:
    """(tên tiếng Anh, hạng mục) của không gian cần render.

    Garden không có loại cụ thể -> lấy default_subject của nhóm. Interior/Exterior
    cũng rơi về default_subject nếu room_type để trống.
    """
    if room_type and room_type != NO_ROOM_TYPE:
        return ROOM_TYPES[room_type]
    return SPACE_TEMPLATES[space]["default_subject"]


def must_have_of(space: str, room_type=None) -> str:
    """Hạng mục định danh, dùng để nhắc lại ở prompt_2."""
    if room_type in ROOM_MUST_HAVE:
        return ROOM_MUST_HAVE[room_type]
    _, items = subject_of(space, room_type)
    return ", ".join(item.strip() for item in items.split(",")[:2])


def _target_words(space: str, room_type=None) -> set:
    room_en, items = subject_of(space, room_type)
    return _words(room_en) | _words(items) | _words(ROOM_EXTRA.get(room_type or "", ""))


def wrong_furniture_for(space: str, room_type=None) -> str:
    """Đồ trong nhà dễ bị giữ lại. Bỏ những món mà chính loại đích cũng cần.

    Render phòng ngủ -> bỏ "bed" khỏi negative (đích cần "double bed").
    Chỉ dùng cho Interior.
    """
    if space != SPACE_INTERIOR:
        return ""
    target = _target_words(space, room_type)
    return ", ".join(item.strip() for item in WRONG_FURNITURE.split(",")
                     if item.strip() and not (_words(item) & target))


def wrong_rooms_for(space: str, room_type=None) -> str:
    """Tên các loại khác TRONG CÙNG NHÓM, để đẩy vào negative prompt.

    Đây là thứ dập prior "living room" của SDXL - render phòng tắm mà không cấm
    "living room" thì ảnh ra cứ lãng đãng kiểu phòng khách ốp gạch.
    Chỉ bật cho Interior (xem negate_siblings).
    """
    if not SPACE_TEMPLATES[space]["negate_siblings"] or not room_type:
        return ""
    target = _target_words(space, room_type) - ROOM_NAME_STOPWORDS
    others = []
    for name in SPACES[space]:
        if name == room_type:
            continue
        short = ROOM_TYPES[name][0].replace(" interior", "").replace(" exterior", "")
        if (_words(short) - ROOM_NAME_STOPWORDS) & target:
            continue
        others.append(short)
    return ", ".join(others)


# ---------- 3.6 Ghép prompt ----------
def resolve_color(space: str, color: str, style: str, seed: int = None) -> tuple:
    """Trả về (tên bảng màu dùng thật, đoạn mô tả màu).

    "Theo style"  -> lấy màu mặc định của style
    "Surprise Me" -> random trong CÁC BẢNG CỦA NHÓM ĐÓ, khoá theo seed nên cùng seed
                     ra cùng màu, tái tạo lại được.
    """
    if color == COLOR_BY_STYLE:
        return f"{COLOR_BY_STYLE} ({style})", STYLE_PROMPTS[style][1]
    pool = [c for c in COLOR_BY_SPACE[space] if c != COLOR_RANDOM]
    if color == COLOR_RANDOM:
        picked = random.Random(seed).choice(pool)
        return f"{COLOR_RANDOM} -> {picked}", COLOR_PALETTES[picked]
    assert color in COLOR_PALETTES, f"Color không hợp lệ với {space}. Chọn 1 trong: {colors_of(space)}"
    return color, COLOR_PALETTES[color]


def build_prompt(space: str, room_type, style: str,
                 color: str = COLOR_BY_STYLE, seed: int = None) -> tuple:
    """Trả về (prompt, prompt_2).

    SDXL có 2 text encoder, mỗi cái chỉ nhận 77 token. Nhồi hết vào 1 chỗ thì
    diffusers cắt phần cuối -> mất luôn quality tag. Nên tách:
      prompt   = không gian + hạng mục + look của style + bảng màu
      prompt_2 = không gian + hạng mục định danh + chi tiết riêng + tag hoàn thiện + chất lượng

    KHÔNG GIAN PHẢI ĐỨNG ĐẦU. CLIP đánh trọng số token đầu cao hơn hẳn: nếu để style
    lên trước thì "bathroom interior" rơi xuống vị trí ~45 và bị style lấn.
    """
    assert space in SPACES, f"Nhóm không hợp lệ. Chọn 1 trong: {SPACE_OPTIONS}"
    assert style in STYLE_BY_SPACE[space], (
        f"Style '{style}' không dùng được cho {space}. Chọn 1 trong: {styles_of(space)}")
    tpl = SPACE_TEMPLATES[space]
    room_en, items = subject_of(space, room_type)
    look, _ = STYLE_PROMPTS[style]
    _, color_text = resolve_color(space, color, style, seed)
    tags = NON_PHOTO_TAGS if style in NON_PHOTO_STYLES else tpl["quality"]

    head_phrase = f"{room_en} {tpl['suffix']}".strip()
    prompt = f"{head_phrase}, {items}, {look}, {color_text}"
    prompt_2 = ", ".join(x for x in (f"{room_en} with {must_have_of(space, room_type)}",
                                     ROOM_EXTRA.get(room_type or "", ""),
                                     tpl["complete"], tags) if x)
    return prompt, prompt_2


def build_negative(space: str, room_type=None) -> tuple:
    """Trả về (negative_prompt, negative_prompt_2).

    Ràng buộc nhóm đặt ở encoder 1 (còn dư token); encoder 2 dành cho danh sách dài:
    chưa hoàn thiện + đồ sai + tên các loại cùng nhóm.
    """
    tpl = SPACE_TEMPLATES[space]
    neg_1 = ", ".join(x for x in (NEGATIVE_PROMPT, ROOM_NEGATIVE.get(room_type or "", ""),
                                  tpl["other_space_neg"], tpl["extra_neg"]) if x)
    neg_2 = ", ".join(x for x in (tpl["empty_neg"],
                                  wrong_furniture_for(space, room_type),
                                  wrong_rooms_for(space, room_type)) if x)
    return neg_1, neg_2


def count_tokens(text: str):
    """Đếm token bằng chính tokenizer của pipeline. None nếu chưa load model."""
    _pipe = globals().get("pipe", None)
    return None if _pipe is None else len(_pipe.tokenizer(text).input_ids)


def check_prompt(space: str, room_type, style: str, color: str = COLOR_BY_STYLE,
                 seed: int = None, verbose: bool = True):
    """In prompt + số token, cảnh báo nếu vượt 77 (SDXL sẽ cắt phần vượt)."""
    prompt, prompt_2 = build_prompt(space, room_type, style, color, seed)
    negative, negative_2 = build_negative(space, room_type)
    if verbose:
        print(f"[nhóm] {space}   [màu] {resolve_color(space, color, style, seed)[0]}\n")
    for label, text in (("prompt", prompt), ("prompt_2", prompt_2),
                        ("negative", negative), ("negative_2", negative_2)):
        n = count_tokens(text)
        flag = "  <-- VƯỢT 77 TOKEN, SDXL sẽ cắt phần cuối" if (n or 0) > 77 else ""
        if verbose:
            print(f"[{label}] {'?' if n is None else n} token{flag}\n  {text}\n")
    return prompt, prompt_2


# ---------- 3.7 Tự kiểm tra ----------
# Từ chỉ món đồ gắn với 1 loại cụ thể: style chứa những từ này sẽ nhét đồ phòng khách
# vào bếp / sân vườn, và đụng luôn negative "đồ sai".
STYLE_BANNED_OBJECTS = {
    "sofa", "couch", "upholstery", "bed", "mattress", "headboard", "rug", "curtains",
    "tatami", "fireplace", "bathtub", "toilet", "desk", "artwork", "triptych",
    "dining", "cabinets", "countertop", "island", "wardrobe", "nightstand",
}
# Từ chỉ MÀU: phải nằm ở phần màu mặc định hoặc COLOR_PALETTES, không được ở trong look,
# nếu không thì option Color chọn gì cũng bị look đè lại.
STYLE_BANNED_HUES = {
    "white", "grey", "gray", "charcoal", "beige", "greige", "taupe", "cream", "ivory",
    "olive", "terracotta", "pink", "cyan", "blue", "green", "mustard", "gold", "navy",
    "violet", "peach", "blush", "sage", "ochre", "rust", "teal", "red", "purple",
    "lilac", "lavender", "apricot", "tan", "brown", "azure", "plum", "mauve", "coral",
    "turquoise", "emerald", "jade", "amber", "copper", "bronze", "burgundy", "fuchsia",
}
# Cụm từ làm GIẢM SỐ ĐỒ. Style được phép kiềm chế DECOR, KHÔNG được giảm hạng mục
# định danh - vì loại không gian là quyết định.
STYLE_REDUCER_PHRASES = (
    "very few", "few objects", "sparse decoration", "uncluttered", "hidden storage",
    "built-ins", "minimal decoration", "no furniture", "empty", "bare", "clutter",
    "multifunctional",
)


def audit_styles() -> dict:
    """look không được gọi tên món đồ, cũng không được ghi màu."""
    problems = {}
    for name, (look, _colors) in STYLE_PROMPTS.items():
        words = _words(look)
        found = sorted(words & STYLE_BANNED_OBJECTS) + sorted(words & STYLE_BANNED_HUES)
        if found:
            problems[name] = found
    return problems


def audit_styles_reduce_objects() -> dict:
    """Style nào đang chủ động yêu cầu ít đồ -> sẽ phá loại không gian."""
    return {name: [ph for ph in STYLE_REDUCER_PHRASES if ph in look.lower()]
            for name, (look, _c) in STYLE_PROMPTS.items()
            if any(ph in look.lower() for ph in STYLE_REDUCER_PHRASES)}


def audit_must_have() -> dict:
    """Hạng mục định danh phải thực sự nằm trong danh sách hạng mục của loại đó."""
    problems = {}
    for name in ROOM_MUST_HAVE:
        _, items = ROOM_TYPES[name]
        missing = _words(ROOM_MUST_HAVE[name]) - _words(items) - {
            "a", "and", "with", "of", "the", "rows", "two"}
        if missing:
            problems[name] = sorted(missing)
    return problems


def audit_room_negative() -> dict:
    """ROOM_NEGATIVE không được cấm thứ mà chính loại đó cần."""
    problems = {}
    for name, text in ROOM_NEGATIVE.items():
        overlap = _words(text) & (_target_words(SPACE_OF[name], name) - ROOM_NAME_STOPWORDS)
        if overlap:
            problems[name] = sorted(overlap)
    return problems


def audit_room_extra() -> dict:
    """ROOM_EXTRA không được ghi màu, nếu không sẽ đè lên bảng màu người dùng chọn."""
    return {name: sorted(_words(text) & STYLE_BANNED_HUES)
            for name, text in ROOM_EXTRA.items() if _words(text) & STYLE_BANNED_HUES}


def audit_keys() -> dict:
    """Mọi bảng phụ và mọi danh sách theo nhóm phải trỏ tới thứ có thật."""
    problems = {}
    for label, table in (("ROOM_EXTRA", ROOM_EXTRA), ("ROOM_NEGATIVE", ROOM_NEGATIVE),
                         ("ROOM_MUST_HAVE", ROOM_MUST_HAVE)):
        unknown = [k for k in table if k not in ROOM_TYPES]
        if unknown:
            problems[label] = unknown
    for space in SPACES:
        unknown = [s for s in STYLE_BY_SPACE[space] if s not in STYLE_PROMPTS]
        if unknown:
            problems[f"STYLE_BY_SPACE[{space}]"] = unknown
        unknown = [c for c in COLOR_BY_SPACE[space]
                   if c != COLOR_RANDOM and c not in COLOR_PALETTES]
        if unknown:
            problems[f"COLOR_BY_SPACE[{space}]"] = unknown
    return problems


_bad = {
    "style chứa tên đồ hoặc màu": audit_styles(),
    "ROOM_EXTRA chứa màu (sẽ đè lên palette)": audit_room_extra(),
    "ROOM_NEGATIVE cấm thứ chính loại đó cần": audit_room_negative(),
    "style yêu cầu ít đồ (phá loại không gian)": audit_styles_reduce_objects(),
    "ROOM_MUST_HAVE đòi món không có trong ROOM_TYPES": audit_must_have(),
    "tham chiếu tới thứ không tồn tại": audit_keys(),
}
if any(_bad.values()):
    for _label, _items in _bad.items():
        for _name, _detail in _items.items():
            print(f"CẢNH BÁO [{_label}] {_name}: {_detail}")
else:
    print("Tự kiểm tra OK: style không chứa tên đồ/màu và không giảm số đồ, ROOM_EXTRA "
          "không chứa màu, ROOM_NEGATIVE không cấm thứ chính loại đó cần, "
          "mọi tham chiếu đều hợp lệ.")

for _space in SPACE_OPTIONS:
    _n = len(rooms_of(_space)) or "không có (chỉ style + màu)"
    print(f"  {_space:9s}: {_n} loại | {len(styles_of(_space))} style | {len(colors_of(_space))} tuỳ chọn màu")
print(f"Tổng: {len(STYLE_PROMPTS)} style, {len(COLOR_PALETTES)} bảng màu (lấy từ 6 file JSON)")
print("\nVí dụ - Garden (không có loại), style Balinese, màu Tropic Bloom:\n")
check_prompt(SPACE_GARDEN, None, "Balinese", "Tropic Bloom")


## 4. Hàm xử lý chính

Chỉ còn **một cách xử lý**, điều khiển bằng đúng 1 công tắc `keep_layout`:

| | Control map | Weight | Kết quả |
|---|---|---|---|
| `keep_layout=True` | MLSD | 0.60 | Cửa sổ, chân tường, góc tường giữ đúng chỗ. Đồ đạc cũ vẫn bị xoá. |
| `keep_layout=False` | MLSD | 0.25 | Chỉ giữ đại khái khối phòng, model tự do đổi mảng tường / vị trí cửa sổ. |

**Vì sao dùng MLSD chứ không phải Canny hay Depth:** Canny bắt *mọi* đường biên nên giữ luôn
đường viền cái sofa cũ — model sẽ nhồi đồ mới vào đúng hình sofa đó, ra đồ méo. Depth thì giữ
nguyên khối 3D của đồ cũ, còn tệ hơn. MLSD chỉ bắt **đường thẳng dài** = gần như chỉ còn kiến trúc,
nên xoá đồ cũ rồi bày lại từ đầu mới sạch.

*Hạn chế:* đồ nội thất cũng có vài cạnh thẳng (mặt bàn, lưng sofa). Nếu ảnh ra vẫn còn dấu vết đồ cũ
thì nâng `mlsd_threshold` ở cell nâng cao lên 0.2-0.3.

Ảnh luôn được resize **giữ đúng tỉ lệ gốc** về ~1 MP (SDXL train quanh mức này), không bóp về vuông.


In [ ]:
# Thông số mặc định. Sửa ở cell "Thông số nâng cao" bên dưới, hoặc cứ để nguyên.
ADVANCED = {
    "line_scale_keep": 0.60,   # weight MLSD khi keep_layout = True
    "line_scale_free": 0.25,   # weight MLSD khi keep_layout = False
    "mlsd_threshold": 0.10,    # cao hơn = ít đường hơn = xoá đồ cũ sạch hơn
    "guidance_scale": 6.0,
    "steps": 30,
    "seed": 42,
    "target_px": 1024 * 1024,
}


def fit_size(w: int, h: int, target_px: int) -> tuple:
    """Giữ tỉ lệ gốc, scale về ~target_px, làm tròn về bội số 8 (yêu cầu của SDXL)."""
    ar = w / h
    new_h = (target_px / ar) ** 0.5
    new_w = ar * new_h
    return (max(512, int(round(new_w / 8) * 8)), max(512, int(round(new_h / 8) * 8)))


def make_mlsd(image: Image.Image, thr_v: float = 0.1, thr_d: float = 0.1) -> Image.Image:
    """Chỉ đường THẲNG DÀI -> gần như chỉ còn kiến trúc, đồ đạc bị loại khỏi control map."""
    out = mlsd_detector(
        image, thr_v=thr_v, thr_d=thr_d,
        detect_resolution=512, image_resolution=max(image.size),
    )
    return out.convert("RGB").resize(image.size, Image.LANCZOS)


def redesign_room(input_image_path: str, space_type: str, room_type, style: str,
                  color: str = COLOR_BY_STYLE, keep_layout: bool = True, **over):
    """Render lại theo space_type + room_type + style. Đồ trong ảnh gốc bị xoá hết.

    space_type: "Interior" / "Exterior" / "Garden".
    room_type:  loại cụ thể trong nhóm đó, để None với Garden (nhóm này không có loại).

    color: tên bảng màu trong COLOR_OPTIONS. "Theo style" = dùng màu mặc định của style.
    keep_layout: True = khoá cửa sổ / đường tường theo ảnh gốc, False = cho phép đổi.
    **over: ghi đè bất kỳ khoá nào trong ADVANCED cho riêng lần gọi này.
    """
    cfg = {**ADVANCED, **over}
    line_scale = cfg.get("line_scale") or (
        cfg["line_scale_keep"] if keep_layout else cfg["line_scale_free"])

    src = Image.open(input_image_path).convert("RGB")
    size = fit_size(*src.size, target_px=cfg["target_px"])
    room_image = src.resize(size, Image.LANCZOS)

    line_image = make_mlsd(room_image, thr_v=cfg["mlsd_threshold"])

    # Pipeline được nạp với 1 hoặc 2 ControlNet -> phải truyền đủ ảnh cho từng cái.
    # Depth (nếu có nạp) luôn để weight 0 vì cách làm này không giữ đồ đạc cũ.
    control_images = [line_image]
    control_scales = [line_scale]
    if "depth" in CONTROL_ORDER:
        control_images.append(Image.new("RGB", size, (0, 0, 0)))
        control_scales.append(0.0)

    prompt, prompt_2 = build_prompt(space_type, room_type, style, color, cfg["seed"])
    negative, negative_2 = build_negative(space_type, room_type)

    result = pipe(
        prompt=prompt,
        prompt_2=prompt_2,
        negative_prompt=negative,
        negative_prompt_2=negative_2,
        image=control_images,
        controlnet_conditioning_scale=control_scales,
        num_inference_steps=cfg["steps"],
        guidance_scale=cfg["guidance_scale"],
        width=size[0],
        height=size[1],
        generator=torch.Generator(device="cpu").manual_seed(cfg["seed"]),
    ).images[0]

    return room_image, line_image, result


## 5. Thông số nâng cao — bỏ qua được

Cell này chỉ ghi đè `ADVANCED`. **Không chạy cũng được**, mặc định đã hợp lý.
Chỉ mở ra khi ảnh kết quả có vấn đề cụ thể:

| Triệu chứng | Sửa |
|---|---|
| Ảnh còn dấu vết đồ cũ (sofa mờ trong phòng ngủ) | `mlsd_threshold` lên **0.20-0.30** |
| Đổi sang phòng tắm / bếp / văn phòng mà ảnh vẫn "la lá phòng khách" | đặt `keep_layout = False`, và `mlsd_threshold` lên **0.25** — hình học phòng khách trong ảnh gốc đang lấn |
| Tường / cửa sổ bị méo, lệch | `line_scale_keep` lên **0.75-0.85** |
| Phòng vẫn ít đồ | `line_scale_keep` xuống **0.40-0.50** |
| Ảnh trông "AI", màu bệt, viền gắt | `guidance_scale` xuống **4.5-5.5** và `steps` lên **40** |
| `CUDA out of memory` | `resolution` sang **0.6 MP** |
| Muốn xem phương án bố trí đồ khác | đổi `seed` |
| Chọn `Surprise Me` mà muốn bảng màu khác | đổi `seed` — palette khoá theo seed để lặp lại được |
| Màu ra không giống bảng màu đã chọn | style đang có màu riêng lấn — chạy bước 3 xem cảnh báo tự kiểm tra |


In [ ]:
#@title Thông số nâng cao (không cần sửa) { display-mode: "form" }

#@markdown **`mlsd_threshold`** — ngưỡng nhận đường thẳng. Cao hơn = giữ ít đường hơn = **xoá đồ cũ sạch hơn**, nhưng quá cao thì mất cả đường kiến trúc và phòng bắt đầu méo.
mlsd_threshold = 0.1  #@param {type:"slider", min:0.05, max:0.4, step:0.05}
#@markdown **`line_scale_keep`** — mức bám kiến trúc khi `keep_layout = True`. Cao = tường/cửa sổ đúng chỗ tuyệt đối nhưng model bị bó, ít đồ hơn.
line_scale_keep = 0.6  #@param {type:"slider", min:0.2, max:1.0, step:0.05}
#@markdown **`line_scale_free`** — mức bám khi `keep_layout = False`. Để thấp cho model tự do đổi layout.
line_scale_free = 0.25  #@param {type:"slider", min:0.0, max:0.6, step:0.05}
#@markdown **`guidance_scale` (CFG)** — mức tuân prompt. RealVisXL/Juggernaut thích **4-7**; đẩy lên 10-12 là ảnh cháy màu, trông "AI" ngay.
guidance_scale = 6.0  #@param {type:"slider", min:3.0, max:10.0, step:0.5}
#@markdown **`steps`** — số bước khử nhiễu. 25-30 là đủ, 40+ nét hơn chút nhưng lâu gấp rưỡi.
steps = 30  #@param {type:"integer"}
#@markdown **`seed`** — cùng seed + cùng thông số = ra đúng ảnh cũ. Giữ nguyên khi đang tinh chỉnh để so sánh công bằng, đổi khi muốn phương án bố trí khác.
seed = 42  #@param {type:"integer"}
#@markdown **`resolution`** — ảnh luôn giữ đúng tỉ lệ gốc, chỉ scale về mức pixel này. Ảnh 16:9 ra ~1368x768 (1 MP) hoặc ~1056x592 (0.6 MP).
resolution = "1.0 MP (chất lượng)"  #@param ["1.0 MP (chất lượng)", "0.6 MP (nhanh)"]

ADVANCED.update({
    "mlsd_threshold": mlsd_threshold,
    "line_scale_keep": line_scale_keep,
    "line_scale_free": line_scale_free,
    "guidance_scale": guidance_scale,
    "steps": steps,
    "seed": seed,
    "target_px": 1024 * 1024 if resolution.startswith("1.0") else int(0.6 * 1024 * 1024),
})
for k, v in ADVANCED.items():
    print(f"  {k} = {v}")


## 6. Upload ảnh & render

Lựa chọn: **nhóm không gian** (Interior / Exterior / Garden) → **loại cụ thể** → **phong cách** → **bảng màu**, cộng công tắc `keep_layout`. Bấm **Choose Files** để chọn ảnh phòng (jpg/png/webp), chọn nhiều ảnh được.


In [ ]:
#@title Upload ảnh và render { display-mode: "form" }

#@markdown **`space_type`** — nhóm không gian, bao trùm `room_type`:
#@markdown - **Interior** — trong nhà: phòng khách, ngủ, ăn, tắm, bếp, game, nhà hàng, văn phòng tại gia, quán cà phê, văn phòng
#@markdown - **Exterior** — ngoài nhà: 18 loại (nhà, biệt thự, căn hộ, khách sạn, cửa hàng, cabin, tháp, nông trại...)
#@markdown - **Garden** — sân vườn: **không có loại cụ thể**, chỉ chọn style + màu
#@markdown
#@markdown Nhóm đổi cả cách viết prompt: Interior dùng `interior / fully furnished / empty room`, Exterior dùng `exterior / finished building / construction site`, Garden dùng `landscape photography / fully planted / bare soil`.
space_type = "Interior"  #@param ["Interior", "Exterior", "Garden"]

#@markdown **`room_type`** — loại cụ thể, **phải thuộc `space_type` ở trên**. Nhóm **Garden không có loại** — chọn `— không áp dụng (Garden) —`. Quyết định **đồ gì / hạng mục gì được đưa vào**: chọn Phòng ngủ thì ra giường / tủ / đèn ngủ, kể cả khi ảnh gốc là phòng khách có sofa. Đồ trong ảnh gốc **bị xoá hết**. (Colab không hỗ trợ dropdown phụ thuộc nhau, nên chọn lệch nhóm sẽ báo lỗi kèm danh sách hợp lệ.)
room_type = "Phòng khách"  #@param ["Phòng khách", "Phòng ngủ", "Phòng ăn", "Phòng tắm", "Bếp", "Phòng chơi game", "Nhà hàng", "Văn phòng tại gia", "Quán cà phê", "Văn phòng", "Nhà", "Hồ bơi", "Nhà siêu nhỏ", "Sân trong", "Khu nghỉ dưỡng", "Gian nhà vòm", "Quán cà phê (ngoài)", "Nông trại", "Biệt thự", "Căn hộ", "Nhà phố", "Văn phòng (ngoài)", "Cửa hàng", "Cabin", "Khách sạn", "Gian hàng", "Nhà tranh", "Tháp", "— không áp dụng (Garden) —"]

#@markdown **`style`** — phong cách. **Mỗi nhóm có danh sách style riêng** (Interior 63 / Exterior 62 / Garden 34, lấy từ file JSON), chọn style không thuộc nhóm sẽ báo lỗi kèm danh sách hợp lệ. Quyết định **màu sắc, vật liệu, ánh sáng** của đồ đạc đó, không đổi loại đồ.
style = "Peaceful"  #@param ["Electric", "Modern", "Peaceful", "Farmhouse", "Clean Bright", "Contemporary", "Fresh Airy", "Eclectic", "Elegant", "Minimalist", "Minimal Tranquil", "Cartoon", "Scandinavian", "Simple Calm", "Bright Soothing", "Cyberpunk", "Rustic", "Compact Calm", "Classic Graceful", "Traditional", "Industrial", "Mid-Century", "Japandi", "Bohemian", "Dry Serene", "Romantic Lush", "Natural Cozy", "Soft Aesthetic", "Luxury", "Tropical", "Gothic", "Modern Dynamic", "Dramatic Timeless", "Grand Traditional", "Charming Cozy", "Raw Modern", "Warm Ornate", "Natural", "Calm Meditative", "Lush Vibrant", "Warm Breezy", "Sleek Structured", "Bright Relaxing", "Colorful", "Clean", "Balanced", "Cozy Homey", "Striking", "Sunny", "Boho Scandinavian", "Deep Peaceful", "Soft Natural Light", "Bright Airy", "Cozy Warm", "Oriental", "Colorful Relaxed", "Wabi-Sabi", "Neo-Classic", "Art Deco", "Rustic Luxury", "Colorful Exotic", "Simple Airy", "Rustic Lively", "Brutalist", "Spanish Revival", "Victorian", "Zen", "Chinese", "Cottage", "French", "Italianate", "Japanese", "Mediterranean", "Colonial", "Mid Century", "Retro", "Cottagecore", "Desert", "English Classic", "Midcentury", "Minimal Nature", "Balinese", "Urban Courtyard", "Vertical+Green", "Coastal", "Forest Retreat", "Balanced Elegant"]

#@markdown **`color`** — bảng màu. **Mỗi nhóm có danh sách riêng** (Interior 17 / Exterior 12 / Garden 13). Đây là thứ quyết định **màu tường, màu vải, tông gỗ**:
#@markdown - **Theo style** — dùng màu mặc định của style (Peaceful ra taupe/olive, Cyberpunk ra neon hồng-xanh).
#@markdown - **Surprise Me** — random 1 trong 16 bảng màu. Khoá theo `seed` nên cùng seed ra cùng màu; muốn màu khác thì đổi seed.
#@markdown - **16 bảng còn lại** — đè lên màu mặc định của style, giữ nguyên vật liệu / hình khối / ánh sáng của style. Ví dụ `Cyberpunk + Twilight Blues` = vẫn bề mặt tối bóng, panel hologram, nhưng tông navy-azure thay vì hồng neon.
color = "Theo style"  #@param ["Theo style", "Surprise Me", "Muted Form", "Millennial Gray", "Cozy Beige", "Earth Calm", "Misty Garden", "Antique Sage", "Ocean Mist", "Twilight Blues", "Velvet Dusk", "Sunbaked Clay", "Terracotta Mirage", "Peach Orchard", "Blush Mood", "Pastel Breeze", "Violet Dream", "Retro Pop", "Urban Veil", "Earthy Harmony", "Earthy Hues", "Sand Serenity", "Terra Coast", "Forest Hues", "Tropic Bloom", "Emerald Gem", "Turquoise Lagoon", "Retro Carnival", "Vintage Pop"]

#@markdown ---
#@markdown **`keep_layout`** — có giữ kiến trúc của ảnh gốc hay không:
#@markdown - **True** — cửa sổ, đường chân tường, góc tường, chiều cao trần **đứng đúng chỗ**. Dùng khi đây là phòng thật của khách và họ cần thấy đúng phòng mình.
#@markdown - **False** — chỉ giữ đại khái hình dạng phòng, model được đổi vị trí cửa sổ và mảng tường. Ảnh thường đẹp hơn nhưng không còn là phòng đó nữa.
#@markdown
#@markdown ⚠️ **Đổi sang loại phòng khác hẳn ảnh gốc thì nên để `False`.** Ảnh gốc là phòng khách rộng, cửa sổ lớn, sàn trống; ép giữ đúng hình học đó rồi đòi ra *phòng tắm* thì kết quả vẫn là một cái sảnh rộng có bồn tắm — vì phòng tắm thật vốn nhỏ và kín. Cùng loại phòng (phòng khách → phòng khách, chỉ đổi style) thì để `True`.
keep_layout = True  #@param {type:"boolean"}

import os, time
from google.colab import files

# Colab không hỗ trợ dropdown phụ thuộc nhau -> kiểm tra bộ ba ở đây, báo lỗi kèm
# danh sách hợp lệ thay vì KeyError khó hiểu.
if room_type == NO_ROOM_TYPE:
    room_type = None
if space_type == SPACE_GARDEN:
    assert room_type is None, (
        f"Nhóm Garden không có loại cụ thể - chọn room_type = '{NO_ROOM_TYPE}'")
else:
    assert room_type is not None, (
        f"Nhóm {space_type} cần chọn loại cụ thể. Các loại: {rooms_of(space_type)}")
    assert SPACE_OF[room_type] == space_type, (
        f"'{room_type}' không thuộc nhóm {space_type} - nó thuộc {SPACE_OF[room_type]}.\n"
        f"Các loại của {space_type}: {rooms_of(space_type)}")
assert style in STYLE_BY_SPACE[space_type], (
    f"Style '{style}' không dùng được cho {space_type}.\nCác style hợp lệ: {styles_of(space_type)}")
assert color in colors_of(space_type), (
    f"Bảng màu '{color}' không dùng được cho {space_type}.\nCác bảng hợp lệ: {colors_of(space_type)}")

print("Chọn 1 hoặc nhiều ảnh phòng để upload...")
uploaded = files.upload()

VALID_EXT = (".jpg", ".jpeg", ".png", ".webp", ".bmp")
image_paths = []
for name in uploaded.keys():
    if not name.lower().endswith(VALID_EXT):
        print(f"  Bỏ qua (không phải ảnh): {name}")
        continue
    path = os.path.join("/content", name)
    with open(path, "wb") as f:
        f.write(uploaded[name])
    image_paths.append(path)

assert image_paths, "Chưa có ảnh nào được upload. Chạy lại cell và chọn file ảnh."

_scale = ADVANCED["line_scale_keep"] if keep_layout else ADVANCED["line_scale_free"]
print(f"\n{space_type} / {room_type or "—"} | {style} | màu: {color} | keep_layout={keep_layout} (mlsd weight={_scale})\n")
check_prompt(space_type, room_type, style, color, ADVANCED["seed"])

results = []   # [(tên file, original, control, output), ...]
for path in image_paths:
    t0 = time.time()
    print(f"Đang xử lý: {os.path.basename(path)} ...")
    original, control, output = redesign_room(
        path, space_type, room_type, style, color=color, keep_layout=keep_layout)
    results.append((os.path.basename(path), original, control, output))
    print(f"  xong sau {time.time() - t0:.0f}s | {output.size[0]}x{output.size[1]}")

input_path = image_paths[-1]
_, original, control, output = results[-1]
print(f"\nXong {len(results)} ảnh.")


## 7. Hiển thị kết quả: gốc / đường kiến trúc (MLSD) / kết quả


In [ ]:
from PIL import Image as PILImage
from IPython.display import display

def show_side_by_side(*images, row_height=384):
    """Ghép ngang, chuẩn hoá theo chiều cao để không méo ảnh."""
    imgs = []
    for im in images:
        im = im.convert("RGB")
        w = int(im.width * row_height / im.height)
        imgs.append(im.resize((w, row_height), PILImage.LANCZOS))
    total_width = sum(im.width for im in imgs)
    combined = PILImage.new("RGB", (total_width, row_height), (255, 255, 255))
    x = 0
    for im in imgs:
        combined.paste(im, (x, 0))
        x += im.width
    return combined

for name, orig, ctrl, out in results:
    stem = name.rsplit(".", 1)[0]
    out.save(f"/content/{stem}_result.jpg", quality=95)
    combined = show_side_by_side(orig, ctrl, out)
    combined.save(f"/content/{stem}_compare.jpg", quality=95)
    print(f"{name} -> /content/{stem}_result.jpg  (+ _compare.jpg)")
    display(combined)
    display(out)
